In [22]:
import subprocess
import sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "pandas", "numpy"])

import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

# Set seed so the messy data is reproducible
random.seed(42)
np.random.seed(42)

# --- Generate fake messy eval task data ---

task_ids = [f"TASK-{str(i).zfill(4)}" for i in range(1, 51)]

annotators = ["j.roman", "K.Patel", "m.osei", "K.PATEL", "J.Roman", "d.taylor", "m.Osei"]

statuses = ["pass", "Pass", "PASS", "fail", "Fail", "pending", "PENDING", "escalated", None]

def random_date():
    base = datetime(2025, 1, 1)
    return base + timedelta(days=random.randint(0, 180))

def messy_date(d):
    formats = ["%Y-%m-%d", "%m/%d/%Y", "%d-%m-%Y", "%B %d, %Y"]
    return d.strftime(random.choice(formats))

rows = []
for i in range(80):
    task = random.choice(task_ids)
    rows.append({
        "task_id": task if random.random() > 0.05 else None,
        "annotator": random.choice(annotators),
        "status": random.choice(statuses),
        "score": random.choice([round(random.uniform(0, 1), 2), None, "n/a", 999]),
        "submitted_date": messy_date(random_date()),
        "review_round": random.choice([1, 2, 3, "one", "1st", None])
    })

df_raw = pd.DataFrame(rows)
print(f"Raw dataset: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns")
df_raw.head(10)


Raw dataset: 80 rows, 6 columns


,task_id,annotator,status,score,submitted_date,review_round
0,TASK-0041,d.taylor,Fail,None,2025-01-27,1st
1,NaN,j.roman,fail,0.23,05/24/2025,None
2,TASK-0042,K.PATEL,fail,n/a,01/02/2025,None
3,TASK-0028,K.Patel,fail,n/a,2025-01-27,one
4,TASK-0007,m.osei,Fail,999,2025-05-18,one
5,TASK-0006,m.Osei,pending,0.58,01/12/2025,3
6,TASK-0006,m.Osei,Pass,999,12-06-2025,2
7,TASK-0024,d.taylor,Fail,0.7,06/05/2025,1st
8,TASK-0047,K.PATEL,PENDING,None,25-06-2025,1
9,TASK-0015,m.Osei,pending,0.4,24-02-2025,2


## Step 1: Inspect the Raw Data

Before cleaning anything, we document what's broken and why it matters.

**Issues identified:**
- Annotator names have inconsistent casing (e.g. `K.Patel` vs `K.PATEL`)
- Status labels are not normalized (`pass`, `Pass`, `PASS`)
- Score column contains mixed types: floats, `None`, `"n/a"`, and sentinel value `999`
- Dates appear in at least 4 different formats
- `review_round` mixes integers, ordinals, and written numbers
- Some `task_id` values are missing entirely


In [23]:
# --- Step 1: Normalize annotator names ---
df = df_raw.copy()

df['annotator'] = df['annotator'].str.lower().str.strip()

print("Unique annotators after normalization:")
print(df['annotator'].unique())

Unique annotators after normalization:
<StringArray>
['d.taylor', 'j.roman', 'k.patel', 'm.osei']
Length: 4, dtype: str


In [24]:
# --- Step 2: Normalize status labels ---
df['status'] = df['status'].str.lower().str.strip()

print("Unique statuses after normalization:")
print(df['status'].unique())

Unique statuses after normalization:
<StringArray>
['fail', 'pending', 'pass', 'escalated', nan]
Length: 5, dtype: str


In [25]:
# --- Step 3: Clean the score column ---
def clean_score(val):
    if val is None or val == "n/a":
        return None
    try:
        score = float(val)
        if score == 999:  # sentinel value, not a real score
            return None
        return score
    except:
        return None

df['score'] = df['score'].apply(clean_score)

print("Score column after cleaning:")
print(df['score'].describe())
print(f"\nNull scores: {df['score'].isna().sum()}")

Score column after cleaning:
count    20.00000
mean      0.52000
std       0.27747
min       0.01000
25%       0.25000
50%       0.52000
75%       0.76000
max       0.89000
Name: score, dtype: float64

Null scores: 60


In [26]:
# --- Step 4: Normalize dates ---
from dateutil import parser

def clean_date(val):
    try:
        return parser.parse(val).strftime("%Y-%m-%d")
    except:
        return None

df['submitted_date'] = df['submitted_date'].apply(clean_date)

print("Sample dates after normalization:")
print(df['submitted_date'].head(10))


Sample dates after normalization:
0    2025-01-27
1    2025-05-24
2    2025-01-02
3    2025-01-27
4    2025-05-18
5    2025-01-12
6    2025-12-06
7    2025-06-05
8    2025-06-25
9    2025-02-24
Name: submitted_date, dtype: str


In [27]:
# --- Step 5: Normalize review_round ---
def clean_round(val):
    mapping = {"one": 1, "1st": 1, "two": 2, "2nd": 2, "three": 3, "3rd": 3}
    if val is None:
        return None
    try:
        return int(val)
    except:
        return mapping.get(str(val).lower(), None)

df['review_round'] = df['review_round'].apply(clean_round)

print("Unique review rounds after normalization:")
print(sorted(df['review_round'].dropna().unique()))

Unique review rounds after normalization:
[np.float64(1.0), np.float64(2.0), np.float64(3.0)]


In [28]:
# --- Step 6: Flag duplicates and missing task IDs ---
df['is_duplicate'] = df.duplicated(subset=['task_id', 'annotator', 'submitted_date'], keep='first')
df['missing_task_id'] = df['task_id'].isna()

print(f"Duplicate entries flagged: {df['is_duplicate'].sum()}")
print(f"Missing task IDs: {df['missing_task_id'].sum()}")


Duplicate entries flagged: 0
Missing task IDs: 4


In [29]:
# --- Step 7: Output the clean dataset ---
df_clean = df[df['missing_task_id'] == False].copy()
df_clean = df_clean.drop(columns=['is_duplicate', 'missing_task_id'])

print(f"Clean dataset: {df_clean.shape[0]} rows, {df_clean.shape[1]} columns")
print(f"Removed: {df_raw.shape[0] - df_clean.shape[0]} rows with missing task IDs")
df_clean.head(10)

Clean dataset: 76 rows, 6 columns
Removed: 4 rows with missing task IDs


,task_id,annotator,status,score,submitted_date,review_round
0,TASK-0041,d.taylor,fail,NaN,2025-01-27,1.0
2,TASK-0042,k.patel,fail,NaN,2025-01-02,NaN
3,TASK-0028,k.patel,fail,NaN,2025-01-27,1.0
4,TASK-0007,m.osei,fail,NaN,2025-05-18,1.0
5,TASK-0006,m.osei,pending,0.58,2025-01-12,3.0
6,TASK-0006,m.osei,pass,NaN,2025-12-06,2.0
7,TASK-0024,d.taylor,fail,0.70,2025-06-05,1.0
8,TASK-0047,k.patel,pending,NaN,2025-06-25,1.0
9,TASK-0015,m.osei,pending,0.40,2025-02-24,2.0
10,TASK-0042,d.taylor,escalated,NaN,2025-05-03,NaN


In [30]:
# --- Step 8: Export clean dataset ---
df_clean.to_csv("qa_audit_log_clean.csv", index=False)
print("Clean dataset saved to qa_audit_log_clean.csv")

Clean dataset saved to qa_audit_log_clean.csv


# QA Audit Log Cleaner
## Project 1 — EFoley Data Portfolio

This notebook simulates a real-world data cleaning task using a mock AI evaluation audit log.

Raw evaluation data often arrives with inconsistent formatting, duplicate entries, missing fields, and conflicting status labels. This notebook documents the process of identifying and resolving those issues to produce a clean, analysis-ready dataset.

**Skills demonstrated:** data cleaning, deduplication, null handling, type normalization, anomaly flagging

## Summary of Findings

| Issue | Raw Count | Resolution |
|---|---|---|
| Inconsistent annotator casing | 7 variations | Normalized to lowercase |
| Inconsistent status labels | 9 variations | Normalized to lowercase |
| Bad score values (None/n/a/999) | 60 out of 80 | Converted to null |
| Mixed date formats | 4 formats | Standardized to YYYY-MM-DD |
| Mixed review_round types | 5 variations | Converted to integer |
| Missing task IDs | 4 rows | Flagged and removed |

**Final output:** 76 rows, 6 columns, fully normalized and analysis-ready.

**Key finding:** 75% of score values were unusable in the raw data — a critical data quality issue that would silently corrupt any downstream analysis if left uncleaned.
